<a href="https://colab.research.google.com/github/SABLISTER/Amazon-sort/blob/main/Neuron_Decoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title Play me
!pip install mat73
!pip install gdown
import gc
import psutil
import os
import gdown
# Check current memory
process = psutil.Process(os.getpid())
print(f"Current memory: {process.memory_info().rss / 1024 / 1024 / 1024:.2f} GB")

# Delete your large variables
for var in ['mat_data', 'sessions_data', 'R_list', 'L_list', 'dt_si_list',
            'epochs_list', 'R_data', 'L_data', 'R_window', 'L_window',
            'R_avg', 'L_avg', 'X', 'y', 'X_train', 'X_test', 'svc_model', 'train_test_split']:
    if var in globals():
        del globals()[var]
        print(f"Deleted {var}")

# Clear matplotlib figures
import matplotlib.pyplot as plt
plt.close('all')

# Force garbage collection
gc.collect()
print(f"Just performing a little clean up, New memory: {process.memory_info().rss / 1024 / 1024 / 1024:.2f} GB")
import mat73
import numpy as np
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

def download_file_alternative():
    if os.path.exists('data.mat'):
        print("File already exists!")
        return True
    try:
        url = 'https://drive.google.com/uc?id=1b2mnj2cnx8EnaTD9k_pF7KZYkD098flm'
        output = 'data.mat'
        gdown.download(url, output, quiet=False)
        print("Download complete!")
        return True
    except Exception as e:
        print(f"Error downloading file: {e}")
        return False

# Call the function to download if needed
download_file_alternative()

# Load the data
print("Loading data.mat using mat73...")
mat_data = mat73.loadmat('data.mat')

print("Successfully loaded with mat73")
sessions_data = mat_data['data']
print(f"Found data with fields: {list(sessions_data.keys())}")

# Get the lists of R and L data, dt_si, and epochs
R_list = sessions_data['R']
L_list = sessions_data['L']
dt_si_list = sessions_data['dt_si']
epochs_list = sessions_data['epochs']
num_sessions = len(dt_si_list)

# Create widgets
session_dropdown = widgets.Dropdown(
    options=[(f"Session {i+1}", i) for i in range(num_sessions)],
    description='Session:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

# Create a placeholder for neuron selection
neuron_dropdown = widgets.Dropdown(
    options=[],
    description='Neuron:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

# Button to run analysis
analyze_button = widgets.Button(
    description='Analyze Neuron',
    button_style='primary',
    layout=widgets.Layout(width='200px')
)

# Output widget for results
output = widgets.Output()

# Session info display area
session_info = widgets.Output()

# Function to update neuron options when session changes
def update_neuron_options(change):
    with session_info:
        clear_output(wait=True)
        session_idx = change['new']
        try:
            # Get R and L data for this session
            R_data = R_list[session_idx][0]
            L_data = L_list[session_idx][0]

            num_neurons = R_data.shape[1]
            print(f"Session {session_idx+1}:")
            print(f"R shape: {R_data.shape}, L shape: {L_data.shape}")
            print(f"This session has {num_neurons} neurons")

            # Update neuron dropdown options
            neuron_dropdown.options = [(f"Neuron {i}", i) for i in range(num_neurons)]
        except Exception as e:
            print(f"Error loading session {session_idx+1}: {e}")

# Function to run analysis when button is clicked
def run_analysis(b):
    with output:
        clear_output(wait=True)
        session_idx = session_dropdown.value

        try:
            # Get session data
            dt_si = dt_si_list[session_idx]
            if isinstance(dt_si, np.ndarray):
                dt_si = dt_si.item()

            session_epochs = epochs_list[session_idx]

            # Extract go cue time
            if isinstance(session_epochs, dict) and 'cue' in session_epochs:
                go_cue_time = session_epochs['cue']
                if isinstance(go_cue_time, np.ndarray):
                    go_cue_time = go_cue_time.item()
            else:
                # Use default go cue time
                go_cue_time = 4.5

            # Calculate frame indices
            go_frame = int(go_cue_time / dt_si)
            window_frames = int(3.0 / dt_si)  # 3 seconds post-go cue
            time_window = slice(go_frame, go_frame + window_frames)

            # Get R and L data
            R_data = R_list[session_idx][0]  # First element is non-photostim trials
            L_data = L_list[session_idx][0]

            # Check time window bounds
            max_frames = min(R_data.shape[0], L_data.shape[0])
            if go_frame + window_frames > max_frames:
                time_window = slice(go_frame, max_frames)
                print(f"Adjusted time window: {go_frame} to {max_frames}")

            # Extract time window
            R_window = R_data[time_window, :, :]
            L_window = L_data[time_window, :, :]

            # Average over time, then transpose to [trials × neurons]
            R_avg = np.nanmean(R_window, axis=0).T
            L_avg = np.nanmean(L_window, axis=0).T

            # Replace NaNs
            R_avg = np.nan_to_num(R_avg)
            L_avg = np.nan_to_num(L_avg)

            # Stack for this session
            X = np.vstack([R_avg, L_avg])
            y = np.concatenate([np.zeros(R_avg.shape[0]), np.ones(L_avg.shape[0])])

            # Train/test split
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=0.2, random_state=42)

            # Train full decoder - create and fit the model ONCE
            svc_model = SVC(kernel='sigmoid', degree=3, gamma='auto', verbose=True, max_iter=1000)
            svc_model.fit(X_train, y_train)

            # Evaluate
            y_pred = svc_model.predict(X_test)
            accuracy = accuracy_score(y_test, y_pred)
            print(f"Full model accuracy: {accuracy*100:.2f}%")

            # Extract this neuron's activity for each trial
            neuron_activity_R = R_avg[:, neuron_dropdown.value]
            neuron_activity_L = L_avg[:, neuron_dropdown.value]

            # Display statistics for selected neuron
            print(f"\nNeuron {neuron_dropdown.value} Analysis:")
            print(f"Mean activity during right trials: {np.mean(neuron_activity_R):.4f}")
            print(f"Mean activity during left trials: {np.mean(neuron_activity_L):.4f}")

            # Show selectivity
            selectivity = np.mean(neuron_activity_R) - np.mean(neuron_activity_L)
            if abs(selectivity) < 0.01:
                print(f"This neuron shows minimal selectivity ({selectivity:.4f})")
            elif selectivity > 0:
                print(f"This neuron is MORE active during RIGHT trials (selectivity: {selectivity:.4f})")
            else:
                print(f"This neuron is MORE active during LEFT trials (selectivity: {selectivity:.4f})")

            # Analyze feature importance for this neuron
            if svc_model.kernel == 'linear':
                neuron_weight = svc_model.coef_[0][neuron_dropdown.value]
                print(f"Decoder weight: {neuron_weight:.4f}")

                if abs(neuron_weight) < 0.01:
                    print("This neuron has minimal influence on the decoder's predictions")
                elif neuron_weight > 0:
                    print("Positive weight means this neuron's activity increases the probability of RIGHT prediction")
                else:
                    print("Negative weight means this neuron's activity increases the probability of LEFT prediction")
            else:
                # For non-linear SVMs, use feature permutation importance
                print("\nAnalyzing feature importance (non-linear kernel)...")
                # Compute permutation importance for this specific neuron
                baseline_score = svc_model.score(X_test, y_test)

                # Permute values of this specific neuron
                X_test_permuted = X_test.copy()
                np.random.shuffle(X_test_permuted[:, neuron_dropdown.value])
                permuted_score = svc_model.score(X_test_permuted, y_test)

                # The importance is the decrease in accuracy when this feature is shuffled
                importance = baseline_score - permuted_score
                print(f"Neuron {neuron_dropdown.value} importance: {importance:.4f}")

                if abs(importance) < 0.001:
                    print("This neuron has minimal influence on the decoder's predictions")
                elif importance > 0:
                    print("Permuting this neuron's values significantly hurts performance - it's important for prediction")
                else:
                    print("Permuting this neuron's values actually improves performance - it may be adding noise")

            # Plot the distribution of activity for this neuron
            plt.figure(figsize=(12, 5))

            # Histogram of activity by trial type
            plt.subplot(1, 2, 1)
            plt.hist(neuron_activity_R, alpha=0.7, bins=15, label='Right Trials', color='blue')
            plt.hist(neuron_activity_L, alpha=0.7, bins=15, label='Left Trials', color='red')
            plt.xlabel('Activity Level')
            plt.ylabel('Number of Trials')
            plt.title(f'Neuron {neuron_dropdown.value} Activity Distribution')
            plt.legend()

            # Single neuron decoder
            plt.subplot(1, 2, 2)

            # Train a single neuron decoder
            X_single = X[:, [neuron_dropdown.value]]  # Keep as 2D array with one feature

            # Split with same random seed for consistency
            X_train_single, X_test_single, y_train_single, y_test_single = train_test_split(
                X_single, y, test_size=0.2, random_state=42)

            # Train and evaluate
            clf_single = LogisticRegression(solver='liblinear', max_iter=1000)
            clf_single.fit(X_train_single, y_train_single)
            y_pred_single = clf_single.predict(X_test_single)
            single_accuracy = accuracy_score(y_test_single, y_pred_single)

            # Plot the decision boundary
            plt.scatter(X_test_single[y_test_single==0], [0.1]*np.sum(y_test_single==0), c='blue', marker='o', label='Right Trials')
            plt.scatter(X_test_single[y_test_single==1], [0.2]*np.sum(y_test_single==1), c='red', marker='o', label='Left Trials')

            # Mark incorrect predictions
            incorrect = y_pred_single != y_test_single
            plt.scatter(X_test_single[incorrect], [0.3]*np.sum(incorrect), s=100, facecolors='none', edgecolors='black', label='Incorrect')

            # Add vertical line for decision boundary
            if abs(clf_single.coef_[0][0]) > 1e-6:  # Avoid division by zero
                boundary = -clf_single.intercept_[0] / clf_single.coef_[0][0]
                plt.axvline(x=boundary, color='black', linestyle='--', label='Decision Boundary')

            plt.title(f'Single Neuron Decoder (Accuracy: {single_accuracy*100:.1f}%)')
            plt.xlabel('Neuron Activity')
            plt.yticks([])
            plt.legend()

            plt.tight_layout()
            plt.show()

            print(f"\nSingle neuron decoder accuracy: {single_accuracy*100:.2f}%")
            print(f"Full model accuracy: {accuracy*100:.2f}%")

            # Compare with full model predictions
            print("\nPrediction analysis for example trials:")
            for i in range(min(5, len(X_test_single))):
                single_pred = "RIGHT" if y_pred_single[i] == 0 else "LEFT"
                full_pred = "RIGHT" if y_pred[i] == 0 else "LEFT"
                true_type = "RIGHT" if y_test_single[i] == 0 else "LEFT"

                single_correct = "✓" if y_pred_single[i] == y_test_single[i] else "✗"
                full_correct = "✓" if y_pred[i] == y_test_single[i] else "✗"

                activity = X_test_single[i][0]

                print(f"Trial {i+1}: Activity = {activity:.4f}")
                print(f"  Single neuron predicts: {single_pred} {single_correct}")
                print(f"  Full model predicts:    {full_pred} {full_correct}")
                print(f"  True trial type:        {true_type}")

        except Exception as e:
            print(f"Error analyzing neuron: {e}")
            import traceback
            traceback.print_exc()

# Removed  analyze_neuron_importance it got funky
# The code now handles both linear and non-linear cases directly in run_analysis

# Connect event handlers
session_dropdown.observe(update_neuron_options, names='value')
analyze_button.on_click(run_analysis)

# Initial update of neuron options
update_neuron_options({'new': session_dropdown.value})

# Layout
controls = widgets.VBox([
    widgets.HTML(value="<h3>Neural Decoder Analysis</h3>"),
    widgets.HBox([session_dropdown, neuron_dropdown, analyze_button]),
    session_info
])

# Display UI
display(controls)
display(output)

Current memory: 0.12 GB
Just performing a little clean up, New memory: 0.12 GB


Downloading...
From (original): https://drive.google.com/uc?id=1b2mnj2cnx8EnaTD9k_pF7KZYkD098flm
From (redirected): https://drive.google.com/uc?id=1b2mnj2cnx8EnaTD9k_pF7KZYkD098flm&confirm=t&uuid=c9fc9e1d-9585-4886-8032-9991b0b109c8
To: /content/data.mat
100%|██████████| 6.25G/6.25G [01:28<00:00, 70.4MB/s]


Download complete!
Loading data.mat using mat73...
Successfully loaded with mat73
Found data with fields: ['CL', 'CR', 'L', 'R', 'XY', 'distance', 'dt_si', 'epochs', 'file', 'stimXY']


Output()